# Day 7 — Weekly Project: House Price Prediction

## 1. Learning Objectives
- Synthesize all knowledge from Week 1.
- Build a complete end-to-end Machine Learning pipeline.
- Implement Data Preprocessing, Splitting, and Training.
- Evaluate model performance on unseen data.

## 2. Project Overview
You are hired as a Data Scientist by a Real Estate firm. They want you to predict house prices based on multiple features (Square footage, number of bedrooms, and age of the house). You will build a Multiple Linear Regression model from scratch using PyTorch.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

## Step 1: Create the Dataset
In the real world, you'd load this from a CSV using pandas. Here, we'll simulate the dataset.

In [ ]:
# Features: [Sqft (in 1000s), Bedrooms, Age (in years)]
torch.manual_seed(42)
num_samples = 500
X = torch.rand(num_samples, 3)
X[:, 0] = X[:, 0] * 3 + 1   # Sqft: 1000 to 4000
X[:, 1] = X[:, 1] * 4 + 1   # Bedrooms: 1 to 5
X[:, 2] = X[:, 2] * 50      # Age: 0 to 50

# True weights: Sqft adds $150k, Bedroom adds $25k, Age subtracts $2k per year. Base price: $50k.
true_w = torch.tensor([[150.0], [25.0], [-2.0]])
true_b = 50.0

# Add some random market noise
y = X @ true_w + true_b + torch.randn(num_samples, 1) * 15.0

print("Features shape:", X.shape)
print("Target shape:", y.shape)

## Step 2: Data Preprocessing (Normalization)
**Crucial Concept:** If you try to train a model where one feature is 4000 and another is 2, the gradients will explode or vanish. We must normalize the features so they are all roughly on the same scale (e.g., between 0 and 1, or mean 0 / std 1).

In [ ]:
# Normalize features to have mean 0 and standard deviation 1 (Standardization)
X_mean = X.mean(dim=0)
X_std = X.std(dim=0)
X_norm = (X - X_mean) / X_std

print("Normalized X mean (should be ~0):", X_norm.mean(dim=0))
print("Normalized X std (should be ~1):", X_norm.std(dim=0))

## Step 3: Train / Validation / Test Split
We never evaluate our model on the data it trained on. 
- **Train (70%)**: To learn the weights.
- **Validation (15%)**: To tune hyperparameters (like learning rate).
- **Test (15%)**: To report final performance.

In [ ]:
train_size = int(0.7 * num_samples)
val_size = int(0.15 * num_samples)

X_train = X_norm[:train_size]
y_train = y[:train_size]

X_val = X_norm[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X_norm[train_size + val_size:]
y_test = y[train_size + val_size:]

print("Train samples:", X_train.shape[0])

## Step 4: Model & Training Loop
**Your turn!** Complete the training loop below.

In [ ]:
# 1. Initialize weights (3 features) and bias. Ensure requires_grad=True.
w = torch.randn(3, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.1
epochs = 200
train_losses = []
val_losses = []

for epoch in range(epochs):
    # --- TRAIN ---
    # 1. Forward Pass
    y_pred = X_train @ w + b
    
    # 2. Calculate MSE Loss
    train_loss = ((y_pred - y_train) ** 2).mean()
    train_losses.append(train_loss.item())
    
    # 3. Backward Pass
    train_loss.backward()
    
    # 4. Update Weights and Zero Gradients
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
        
    # --- VALIDATION ---
    # Calculate validation loss without tracking gradients
    with torch.no_grad():
        val_pred = X_val @ w + b
        val_loss = ((val_pred - y_val) ** 2).mean()
        val_losses.append(val_loss.item())
        
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Train Loss: {train_loss.item():.2f} | Val Loss: {val_loss.item():.2f}")

## Step 5: Visualization & Evaluation
Let's see if the model learned without overfitting.

In [ ]:
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()

# Final Test Evaluation
with torch.no_grad():
    test_pred = X_test @ w + b
    test_loss = ((test_pred - y_test) ** 2).mean()
    print(f"\nFinal Test Loss (MSE): {test_loss.item():.2f}")
    print(f"Average Error per House: ${torch.sqrt(test_loss).item():.2f}k")

## Week 1 Conclusion
Congratulations! You have successfully built a complete machine learning pipeline using raw PyTorch tensors and Autograd. 

In Week 2 (Phase 2), we will stop doing this manually and introduce `torch.nn`, PyTorch's powerful Neural Network module, which abstracts all of this math into beautiful, clean, reusable code blocks.